# Data Analysis of the YAOKI rover on IM-2
2026/05/11

The graphs produced in this jupyter notebook are analysed in the Mission Report 


Instructions to access data from the Yamcs database (Tested on Ubuntu 24): 
1. Start the Yamcs server: `cd im2-yaoki-yamcs-public/yamcs-server` `mvn yamcs:run`
2. You can open the Yamcs Web interface via your web browser (http://localhost:8090) and check the data in the [Archive Browser tab](http://localhost:8090/archive?c=im2-yaoki__realtime&start=2025-02-25T18:33:56.087Z&stop=2025-03-08T01:31:16.844Z&packets=true&parameters=true&commands=true&events=false) and plot the telemetry items in the [Parameters tab](http://localhost:8090/telemetry/parameters/YAOKI/Lander/temperature0/-/chart?c=im2-yaoki__realtime&interval=CUSTOM&customStart=Sun%20Mar%2002%202025%2008:28:34%20GMT%2B0900%20(Japan%20Standard%20Time)&customStop=Mon%20Jun%2002%202025%2008:43:34%20GMT%2B0900%20(Japan%20Standard%20Time)). 

3. To replicate the graphs install the dependencies: 
```bash
cd im2-yaoki-yamcs-public
python3 -m venv .venv/yaoki
source venv/yaoki/bin/activate
pip install -r analysis/requirements.txt
```
4. and then Open VS Code and run this notebook: `yamcs_archive.ipynb`

In [ ]:
from datetime import datetime, timedelta, timezone
from itertools import islice

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# YAMCS imports
from yamcs.client import YamcsClient

# Configuration
pio.renderers.default = 'vscode'  # Reduce notebook size by not embedding plotly.js
pd.set_option('display.max_colwidth', None)

In [ ]:
client = YamcsClient('localhost:8090')
archive = client.get_archive(instance='im2-yaoki')

In [ ]:
# Mission Timeline Constants
IM2_LAUNCH_DATE = "2025-02-27T00:16:30Z"
IM2_LANDING_DATE = "2025-03-06T17:28:50Z"
IM2_FIRST_TELEMETRY_DATE = "2025-02-26T18:50:58.457Z"
IM2_LAST_TELEMETRY_DATE = "2025-03-07T05:42:56.476Z"

YAOKI_FIRST_TELEMETRY_DATE = '2025-03-07 02:17:15.150000+00:00'  # YAOKI first housekeeping telemetry
YAOKI_LAST_TELEMETRY_DATE = '2025-03-07 04:33:04.389000+00:00'  # YAOKI last received image packet


# Analysis Constants
IMAGE_TOTAL_PACKETS = 960
WIRE_CUT_DURATION_SECONDS = 200
NO_RX_TIMEOUT_MINUTES = 3
TEMPERATURE_LIMIT_CELSIUS = -40
MIN_GAP_SECONDS = 300.0

def adjust_timestamp(timestamp_str: str, delta: timedelta) -> datetime:
    """
    Adjust the provided ISO 8601 timestamp by the given timedelta.
    
    Parameters:
        timestamp_str (str): An ISO 8601 timestamp with 'Z' indicating UTC.
        delta (timedelta): The time delta to adjust.
        
    Returns:
        datetime: The adjusted timestamp as a datetime object.
    """
    if timestamp_str.endswith('Z'):
        timestamp_str = timestamp_str.replace("Z", "+00:00")
    dt = datetime.fromisoformat(timestamp_str)
    return dt + delta

In [ ]:
start = datetime.fromisoformat(YAOKI_FIRST_TELEMETRY_DATE)
stop = datetime.fromisoformat(YAOKI_LAST_TELEMETRY_DATE)
stop_margin = adjust_timestamp(YAOKI_LAST_TELEMETRY_DATE, timedelta(hours=1))  # with margin for data getting
print(stop)
mission_duration = stop - start
print(mission_duration)

Examples getting data from Yamcs database

In [ ]:
param_name = "/YAOKI/Lander/temperature0"
iterable = archive.list_parameter_values(
   param_name, descending=False, start=start, stop=stop
)
for pval in islice(iterable, 0, 10):
    print(pval.name, pval.generation_time.astimezone(tz=timezone.utc), pval.raw_value)

In [ ]:
param_name = "/YAOKI/Rover/last_packet_num"
iterable = archive.list_parameter_values(
   param_name, descending=True
)
for pval in islice(iterable, 0, 10):
    print(pval.name, pval.generation_time.astimezone(tz=timezone.utc), pval.raw_value)

In [ ]:
def get_data_from_archive(archive, dataspec, start, stop):
    """
    Extracts data from the archive and stores it in a DataFrame.

    Parameters:
        archive: The Yamcs archive client instance.
        dataspec: A list of dictionaries specifying sensor names and parameter paths.
        start: The start datetime for data extraction.
        stop: The stop datetime for data extraction.

    Returns:
        pd.DataFrame: A DataFrame containing the extracted data with columns ['sensor_name', 'timestamp', 'value'].
    """
    rows = (
        {
            'sensor_name': spec["sensor_name"],
            'timestamp': pval.generation_time.astimezone(tz=timezone.utc),
            'value': pval.raw_value
        }
        for spec in dataspec
        for pval in archive.list_parameter_values(
            spec["param_path"], descending=False, start=start, stop=stop
        )
    )
    # Create the DataFrame from the generator
    df = pd.DataFrame(rows, columns=['sensor_name', 'timestamp', 'value'])
    return df

## Analyze Temperature Data

In [ ]:
# Define a dataspec for data extraction
dataspec = [
    {
        "sensor_name": f"Rover_{i}", 
        "param_path": f"/YAOKI/Rover/ADC_TEMP{i}"
    } for i in range(8)
] + [
    {
        "sensor_name": f"Lander_{i}", 
        "param_path": f"/YAOKI/Lander/temperature{i}"
    } for i in range(2)
]
df = get_data_from_archive(archive, dataspec, start, stop)

In [ ]:
fig = px.scatter(
    df,
    x='timestamp',
    y='value',
    color='sensor_name',
    title="Yaoki Surface OPS: Temperature Sensors",
    labels={'value': 'Temperature (°C)', 'timestamp': 'Time (UTC)'},
    # template="simple_white"
)
fig.add_hline(
    y=-40,
    line_color='black',
    annotation_text='rover temperature sensor limit',
    annotation_position='bottom'
)
fig.add_vline(
    x=start.astimezone(timezone.utc), 
    line_dash="dash", 
    line_color="black"
    )
fig.add_vline(
    x=stop.astimezone(timezone.utc), 
    line_dash="dash", 
    line_color="black"
    )

fig.update_layout(
    yaxis_range=[-65, 0]
)
fig.update_layout(legend=dict(orientation='h', yanchor='top', y=-0.4))
legend_mapping = {
    
    'Rover_0': 'R0: Body (Bottom)',
    'Rover_1': 'R1: Motor board (middleboard)',
    'Rover_2': 'R2: UHF radio',
    'Rover_3': 'R3: Body (Top)',
    'Rover_4': 'R4: On CPU (Middle)',
    'Rover_5': 'R5: [temperature data not sent]',
    'Rover_6': 'R6: Battery (Bottom)',
    'Rover_7': 'R7: Battery (Top)',
    
    'Lander_0': 'L0: enclosure by burnwire',
    'Lander_1': 'L1: enclosure by mounting bracket'
}
for trace in fig.data:  
    if trace.name in legend_mapping:
        trace.name = legend_mapping[trace.name]
fig.show()

## Analyze Radio data

In [ ]:
# Define a dataspec for data extraction
dataspec = [
    {
        "sensor_name": f"last_rssi", 
        "param_path": f"/YAOKI/Rover/last_rssi"
    },
    {
        "sensor_name": f"Rx_cnt", 
        "param_path": f"/YAOKI/Rover/Rx_cnt"
    },
    {
        "sensor_name": f"RxError_cnt", 
        "param_path": f"/YAOKI/Rover/RxError_cnt"
    },
    {
        "sensor_name": f"Tx_cnt", 
        "param_path": f"/YAOKI/Rover/Tx_cnt"
    },
]
df = get_data_from_archive(archive, dataspec, start, stop)

In [ ]:
fig = px.scatter(
    df,
    x='timestamp',
    y='value',
    color='sensor_name',
    title="Yaoki Surface OPS: Radio RSSI and Telemetry Counters",
    labels={'value': 'Value', 'timestamp': 'Time (UTC)'},
    template="simple_white"
)
fig.add_vline(
    x=start.astimezone(timezone.utc), 
    line_dash="dash", 
    line_color="black",
    line_width=3
)
fig.add_vline(
    x=stop.astimezone(timezone.utc), 
    line_dash="dash", 
    line_color="black",
    line_width=3
)
fig.update_layout(
    yaxis_range=[-100, 110]
)
fig.update_layout(legend=dict(orientation='h', yanchor='top', y=-0.3))
fig.show()

In [ ]:
# outliers:
max_n = 2
filterdf = df[df["sensor_name"] == "last_rssi"]
filterdf.nlargest(max_n, "value").iloc[0:max_n]


## Analyze Camera Data

In [ ]:
# Define a dataspec for data extraction
dataspec = [
    {
        "sensor_name": f"CamSend_num", 
        "param_path": f"/YAOKI/Rover/CamSend_num"
    },
    {
        "sensor_name": f"CamSent_cnt", 
        "param_path": f"/YAOKI/Rover/CamSent_cnt"
    },
]
df = get_data_from_archive(archive, dataspec, start, stop)

In [ ]:
fig = px.scatter(
    df,
    x='timestamp',
    y='value',
    color='sensor_name',
    title="Yaoki Surface OPS: Image Counters",
    labels={'value': 'Value', 'timestamp': 'Time (UTC)'},
    template="simple_white"
)
fig.add_vline(
    x=start.astimezone(timezone.utc), 
    line_dash="dash", 
    line_color="black",
    line_width=3
)
fig.add_vline(
    x=stop.astimezone(timezone.utc), 
    line_dash="dash", 
    line_color="black",
    line_width=3
)
fig.update_layout(legend=dict(orientation='h', yanchor='top', y=-0.2))
fig.show()

In [ ]:
# Define a dataspec for data extraction
dataspec = [
    {
        "sensor_name": f"last_packet_num", 
        "param_path": f"/YAOKI/Rover/last_packet_num",
    },
    {
        "sensor_name": f"missing_packet_cnt", 
        "param_path": f"/YAOKI/Rover/missing_packet_cnt"
    },

]
df = get_data_from_archive(archive, dataspec, start, stop_margin)

In [ ]:
df_filtered = df[df["sensor_name"] == "missing_packet_cnt"]
# 1. sort & re-index
df_filtered = df_filtered.sort_values("timestamp").reset_index(drop=True)

# 2. detect resets
reset_mask = df_filtered["value"].diff() > 0

# 3. previous-row values
prev_vals = df_filtered["value"].shift(1)

# 4. pull the last-before-reset values
last_before_reset = prev_vals[reset_mask].astype(int)

# 5. (optional) their timestamps
last_before_ts = df_filtered["timestamp"].shift(1)[reset_mask]

# 6. assemble into a result
result = pd.DataFrame({
    "timestamp_last_frame"   : last_before_ts.values,
    "missing frames": last_before_reset.values
})
result.index = result.index + 2
# print(result)

# merge rows 9 and 10
result.at[9, 'missing frames'] = result.loc[[9, 10], 'missing frames'].sum()
result = result.drop(index=10)
result.reset_index(drop=True, inplace=True)
result.index = result.index + 2

## statistics of image packets
print("statistics for nominal images:") 
print((result['missing frames'].describe()))
print("")
print(f"median of missing packets: {result['missing frames'].median() / 960 * 100:.2f}%")
print(f"mean of missing packets: {result['missing frames'].mean() / 960 * 100:.2f}%")
print(f"75% of images have less than: {result['missing frames'].quantile(.75) / 960 * 100:.2f}% missing packets")


## adjust the data for the missing images before plotting

# the first image was through the store and forward channel so shift the index
# result.index = result.index + 2

# add first image (via store and forward)
result.loc[1, 'missing frames'] = 18
result.loc[1, 'timestamp_last_frame']   = pd.NaT

# # add missing images at index 13 and 25
def insert_by_label(df, labels, new_row):
    """
    Inserts new_row (a dict of column→value) at each index LABEL in `labels` (these
    are index *labels*, not positions).  Existing rows with index >= LABEL are shifted
    up by 1 each time, so nothing gets overwritten.
    """
    df = df.copy()
    for lbl in sorted(labels):
        # 1) bump every existing label ≥ lbl up by 1
        df.index = pd.Index([i+1 if i >= lbl else i for i in df.index])
        # 2) now there's room at `lbl`, so insert your row
        df.loc[lbl] = new_row
    return df.sort_index()

new_row = {"missing frames": 960, "timestamp_last_frame": pd.NaT}
result = insert_by_label(result, labels=[13, 25], new_row=new_row)

# add last image
last_val = df_filtered['value'].iloc[-1]
last_ts = df_filtered['timestamp'].iloc[-1]
new_idx = result.index.max() + 1
result.loc[new_idx, 'missing frames'] = last_val
result.loc[new_idx, 'timestamp_last_frame']   = last_ts 

result = result.sort_index()
# Ensure consistency of the timestamp column
result['timestamp_last_frame'] = pd.to_datetime(result['timestamp_last_frame'], errors='coerce', utc=True)

result

In [ ]:
fig = px.scatter(
    df,
    x='timestamp',
    y='value',
    color='sensor_name',
    # title="Yaoki Surface OPS: Radio RSSI and Telemetry Counters",
    labels={'value': 'Value', 'timestamp': 'Time (UTC)'},
    template="simple_white"
)
# add a vertical line at each computed last image packet
for ts in result["timestamp_last_frame"].dropna():
    fig.add_vline(
        x=ts,
        line_dash="solid", 
        line_color="orange",
        line_width=3
    )

fig.add_vline(
    x=start.astimezone(timezone.utc), 
    line_dash="dash", 
    line_color="black",
    line_width=3
)
fig.add_vline(
    x=stop.astimezone(timezone.utc), 
    line_dash="dash", 
    line_color="black",
    line_width=3
)
# fig.update_layout(
#     yaxis_range=[-100, 110]
# )
fig.update_layout(legend=dict(orientation='h', yanchor='top', y=-0.2))
fig.show()

In [ ]:
fig = px.bar(
    result,
    y='missing frames',
    text='missing frames',         # ← add the labels
    title="Yaoki Surface OPS: Missing Image Frames",
    labels={'missing frames': 'Missing Frames (log scale)', 'pre_reset_ts': 'Time (UTC)'},
    template="simple_white"
)

colorway = pio.templates['simple_white'].layout.colorway
colors = [colorway[2] if idx  == 1 else colorway[3] if idx in {13, 25} else colorway[1] if idx == 30 else colorway[0] for idx in result.index]
fig.update_traces(marker_color=colors)

fig.update_xaxes(title_text='Capture Image ID')
fig.update_xaxes(
    tickmode='linear',
    tick0=0,
    dtick=1,
    range=[0.5, result.index.max() + 0.5]
)
fig.update_yaxes(type='log', tickformat=".0f", nticks=10)

fig.update_traces(
    marker_color=colors,
    texttemplate='%{text}',        # use the raw value
    textposition='outside',         # place labels above each bar
    cliponaxis=False

)
fig.show()

## Accelerometer Data Analysis

In [ ]:
# Define a dataspec for data extraction
dataspec = [
    {
        "sensor_name": f"Accelerometer_X", 
        "param_path": f"YAOKI/Xr"
    },
    {
        "sensor_name": f"accelerometer_x", 
        "param_path": f"YAOKI/x",
    },
    {
        "sensor_name": f"Accelerometer_Y", 
        "param_path": f"YAOKI/Yr"
    },
    {
        "sensor_name": f"accelerometer_y", 
        "param_path": f"YAOKI/y"
    },
    {
        "sensor_name": f"Accelerometer_Z", 
        "param_path": f"YAOKI/Zr"
    },
    {
        "sensor_name": f"accelerometer_z", 
        "param_path": f"YAOKI/z"
    },
]
df = get_data_from_archive(archive, dataspec, start, stop)

In [ ]:
fig = px.scatter(
    df,
    x='timestamp',
    y='value',
    color='sensor_name',
    title="Yaoki Surface OPS: Accelerometer",
    labels={'value': 'Value', 'timestamp': 'Time (UTC)'},
    template="simple_white"
)
fig.add_vline(
    x=start.astimezone(timezone.utc), 
    line_dash="dash", 
    line_color="black",
    line_width=3
)
fig.add_vline(
    x=stop.astimezone(timezone.utc), 
    line_dash="dash", 
    line_color="black",
    line_width=3
)
# fig.update_layout(
#     yaxis_range=[-100, 110]
# )
# fig.update_yaxes(type='log', tickformat=".1f", nticks=8)  # dont use log for values that are 0 or negative
fig.update_layout(legend=dict(orientation='h', yanchor='top', y=-0.3))
fig.show()

# Status Bits Analysis

In [ ]:
list_parameter_ranges = archive.list_parameter_ranges(
    "/YAOKI/Rover/Status_CAM",
    start=start,
    stop=stop
)
df_ranges = pd.DataFrame([{
    "status": pr.eng_value,   
    "start":  pr.start,      
    "stop":   pr.stop
} for pr in list_parameter_ranges])
df_ranges

In [ ]:
import pandas as pd
import plotly.express as px

def get_status_df(tm_path, start, stop):
    """
    Fetch parameter ranges for a given tm_path from the archive,
    and return a DataFrame with columns: tm, status, start, stop, label.
    """
    # ranges = archive.list_parameter_ranges(tm_path, start=start, stop=stop)
    ranges = archive.list_parameter_ranges(tm_path, start=start, stop=stop, min_gap=300.)
    df = pd.DataFrame([{
        "tm":        tm_path.split('/')[-1],
        "status":        pr.eng_value,
        "start":         pr.start,
        "stop":          pr.stop
    } for pr in ranges])
    df["label"] = df["status"].map({0: "Off", 1: "On"})
    return df

# Fetch and combine into a single DataFrame
tms = [
    "/YAOKI/Rover/Status_BAT",
    "/YAOKI/Rover/VBAT_CHECK",
    "/YAOKI/Rover/Status_ICUT",
    "/YAOKI/Rover/Status_WCUT",
    "/YAOKI/Rover/Status_CAM",
    "/YAOKI/Rover/Status_MOVE",
    "/YAOKI/Rover/Status_NORX",
    "/YAOKI/Rover/Status_TLM",
]
dfs = [get_status_df(path, start, stop) for path in tms]
df_all = pd.concat(dfs, ignore_index=True)
df_all['start'] = pd.to_datetime(df_all['start'], utc=True)
df_all['stop']  = pd.to_datetime(df_all['stop'], utc=True)

# Plot timeline
fig = px.timeline(
    df_all,
    x_start="start",
    x_end="stop",
    y="tm",
    color="label",
    # color_discrete_map={"On": "red", "Off": "green"},
    title="YAOKI Rover Status Timeline",
    labels={"tm": "Parameter", "label": "Status"},
    template="simple_white"
)

# Reverse y-axis so the first tm is on top
fig.update_yaxes(title="", autorange="reversed")
fig.update_xaxes(title="Time (UTC)")
fig.show()

In [ ]:
path = "/YAOKI/Rover/Status_ICUT"
df = get_status_df(path, start, stop)
df

In [ ]:
on_intervals = df[df['status'] == 1]
durations = on_intervals['stop'] - on_intervals['start']
total_on_time = durations.sum()
total_on_time

In [ ]:
data = [
    {"Name": "Status_BAT",    "Description": "Status battery on"},
    {"Name": "VBAT_CHECK",    "Description": "Yaoki power source: 0 if on lander power / 1 if on internal battery power"},
    {"Name": "Status_ICUT",   "Description": "The deployment wire is currently being cut"},
    {"Name": "Status_WCUT",   "Description": "The command to cut the wire was sent AND the time allocated for the wire cut heater has elapsed"},
    {"Name": "Status_CAM",    "Description": "The camera is actively taking the picture / sending the data via CSP"},
    {"Name": "Status_MOVE",   "Description": "The rover should be moving (e.g. time remains since the last move command set the move timer)"},
    {"Name": "Status_NORX",   "Description": "No pings received for 3 minutes"},
    {"Name": "Status_TLM",    "Description": "Telemetry transmission mode: 1 if high frequency / 0 if low frequency"},
]
df = pd.DataFrame(data, columns=["Name", "Description"])
pd.set_option('display.max_colwidth', None)
df = (
    df.style
      .set_table_styles([
          {'selector': 'th', 'props': [('text-align', 'left')]}
      ])
      .set_properties(**{'text-align': 'left'})
)
df

# Telecommand Analysis

In [ ]:
import io
blob = b"".join(archive.export_commands(
    start=start, 
    stop=stop
))
df_cmds = pd.read_csv(io.StringIO(blob.decode('utf-8')), sep='\t')
df_cmds

In [ ]:
import pandas as pd

# Count each telecommand
telecmd_counts = (
    df_cmds['Command Name']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'Command Name', 'Command Name': 'Count'})
)

# Add total row
total = telecmd_counts['count'].sum()
telecmd_counts.loc[len(telecmd_counts)] = ['Total', total]
telecmd_counts



In [ ]:
fig = px.pie(
    df_cmds,
    names='Command Name',
    title='Telecommand Distribution',
    template='simple_white',
    width=800,   # set width in pixels
    height=500   # optionally adjust height
)

# show only the percent (no decimals) inside each slice
fig.update_traces(
    textinfo='percent',
    texttemplate='%{percent:.0%}',
    insidetextorientation='radial'
)

fig.show()

In [ ]:
# show only the commands with the name "/YAOKI/Rover/CutWire"
df_cmds[df_cmds["Command Name"] == "/YAOKI/Rover/CutWire"]
df_cmds[df_cmds["Command Name"] == "/YAOKI/Rover/CaptureCamera"].count()
df_cmds[df_cmds["Command Name"] == "/YAOKI/Rover/CaptureCamera"]
# df_cmds[["Command Name", "Generation Time", "Arguments"]].head(10)

In [ ]:
import re
import pandas as pd
import plotly.express as px

def compute_duration(row):
    cmd  = row['Command Name']
    args = row['Arguments'] or ""
    if cmd == '/YAOKI/Rover/CutWire':
        return pd.Timedelta(seconds=200)
    elif cmd == '/YAOKI/Rover/RotateLeftAndRightMotors':
        nums = re.findall(r'-?\d+', args)
        return pd.Timedelta(seconds=max(abs(int(n)) for n in nums)) if nums else pd.Timedelta(seconds=60)
    else:
        return pd.Timedelta(minutes=1)

def extract_text(row):
    cmd  = row['Command Name']
    args = row['Arguments'] or ""
    # CutWire: print the cut amount
    if cmd == '/YAOKI/Rover/CutWire':
        m = re.search(r'-?\d+', args)
        return m.group() if m else ""
    # Rotate: print max motor value
    if cmd == '/YAOKI/Rover/RotateLeftAndRightMotors':
        nums = re.findall(r'-?\d+', args)
        return str(max(abs(int(n)) for n in nums)) if nums else ""
    # CaptureCamera: only label store_and_forward as SnF
    if cmd == '/YAOKI/Rover/CaptureCamera':
        return "SnF" if args == "cam: store_and_forward" else ""
    return ""

# build timeline DataFrame
df_timeline = df_cmds.copy()
df_timeline['start']    = pd.to_datetime(df_timeline['Generation Time'])
df_timeline['duration'] = df_timeline.apply(compute_duration, axis=1)
df_timeline['end']      = df_timeline['start'] + df_timeline['duration']
df_timeline['text']     = df_timeline.apply(extract_text, axis=1)
df_timeline['cmd']      = df_timeline['Command Name'].str.replace(r'^/YAOKI/Rover/', '', regex=True)

fig = px.timeline(
    df_timeline,
    x_start='start',
    x_end='end',
    y='cmd',
    color='cmd',
    title='Yaoki Surface OPS: Telecommand Execution Timeline',
    labels={'start':'Start Time (UTC)','end':'End Time (UTC)','cmd':'Command'},
    template='simple_white'
)
# annotate above each bar
for _, row in df_timeline.iterrows():
    if row['text']:
        fig.add_annotation(
            x=row['end'],
            y=row['cmd'],
            text=row['text'],
            showarrow=False,
            xanchor='right',
            yanchor='top',
            font=dict(size=10, color='black'),
            yshift=17
        )
fig.update_xaxes(title='Time (UTC)')
# fig.update_yaxes(title='Command Type')
fig.update_layout(margin=dict(t=80, b=50, l=50, r=20))
fig

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# status bits dataframe
dfs = [get_status_df(path, start, stop) for path in tms]
df_all = pd.concat(dfs, ignore_index=True)

# ensure UTC
df_all['start'] = pd.to_datetime(df_all['start'], utc=True)
df_all['stop']  = pd.to_datetime(df_all['stop'],  utc=True)
df_cmds['Generation Time'] = pd.to_datetime(df_cmds['Generation Time'], utc=True)

# build the base figure
fig = px.timeline(
    df_all,
    x_start="start",
    x_end="stop",
    y="tm",
    color="label",
    title="YAOKI Rover Status + Telecommands",
    labels={"tm":"Parameter","label":"Status"},
    template="simple_white"
)

# make status bars 80% transparent
fig.update_traces(opacity=0.2, selector=dict(type="bar"))
fig.update_yaxes(title="", autorange="reversed")
fig.update_xaxes(title="Time (UTC)")

# add a hidden secondary y‐axis that runs 0→1 over the same plot
fig.update_layout(
    yaxis2=dict(
        anchor='x',
        overlaying='y',
        side='right',
        range=[0, 1],
        showticklabels=False,
        showgrid=False
    ),
    margin=dict(t=80, b=40, l=50, r=20)
)

# pick colors from the template
colorway = pio.templates['simple_white'].layout.colorway
cmds = df_cmds['Command Name'].unique()
color_map = {cmd: colorway[i % len(colorway)] for i, cmd in enumerate(cmds)}

# overlay each command type as a Scatter on yaxis2
for cmd in cmds:
    group = df_cmds[df_cmds['Command Name']==cmd]
    # flatten times into line‐segments
    x_vals, y_vals = [], []
    for t in group['Generation Time']:
        x_vals += [t, t, None]
        y_vals += [0, 1, None]
    fig.add_trace(go.Scatter(
        x=x_vals,
        y=y_vals,
        mode='lines',
        line=dict(color=color_map[cmd], width=2),
        name=cmd,
        yaxis='y2'          # <-- send this trace to yaxis2
    ))

fig.update_layout(
    legend=dict(
        orientation="h", 
        y=-0.3, 
        x=1, 
        xanchor='right'
        ),
)
fig

# Lander Telemetry Analysis

In [ ]:
# Define a dataspec for data extraction
dataspec = [
    {
        "sensor_name": f"Temperature0", 
        "param_path": f"/YAOKI/Lander/temperature0"
    },
    {
        "sensor_name": f"Temperature1", 
        "param_path": f"/YAOKI/Lander/temperature1",
    },
]
df = get_data_from_archive(
    archive, 
    dataspec, 
    start=adjust_timestamp(IM2_LAUNCH_DATE, delta=timedelta(hours=-1)), 
    stop=stop_margin
    )
df

In [ ]:
fig = px.scatter(df, x='timestamp', y='value', color='sensor_name',
              title='Temperature Telemetry over Time',
            template="simple_white"
)
fig.show()

In [ ]:
# Clean the data
# Sort the DataFrame by sensor_name and timestamp to ensure correct order.
df_sorted = df.sort_values(['sensor_name', 'timestamp'])

# For each sensor, drop rows where the Temperature is identical to the previous one.
df_clean = df_sorted[
    df_sorted['value'].ne(df_sorted.groupby('sensor_name')['value'].shift())
].reset_index(drop=True)



In [ ]:
fig = px.scatter(df_clean, x='timestamp', y='value', color='sensor_name',
              title='Temperature Telemetry over Time (cleaned)',
                template="simple_white"
)
fig.add_vrect(
    x0=YAOKI_FIRST_TELEMETRY_DATE,
    x1=YAOKI_LAST_TELEMETRY_DATE,
    fillcolor="LightGray",
    opacity=0.3,
    layer="below",
    line_width=0,
)
# 2) Add a label centered on that rectangle
midpoint = (
    pd.to_datetime(YAOKI_FIRST_TELEMETRY_DATE).value
    + pd.to_datetime(YAOKI_LAST_TELEMETRY_DATE).value
) / 2
mid_datetime = pd.to_datetime(midpoint)

fig.add_annotation(
    x=mid_datetime,
    yref="paper",
    y=0.95,  # just below the top of the plot area
    text="YAOKI surface OPS",
    showarrow=False,
    font=dict(color="black", size=12),
    bgcolor="rgba(255,255,255,0.7)"
)

# 3) Add vertical lines for IM2 launch and landing
fig.add_vline(
    x=IM2_LAUNCH_DATE,
    line_width=2,
    # line_dash="dash",
    # line_color="blue"
)
fig.add_annotation(
    x=IM2_LAUNCH_DATE,
    yref="paper",
    y=1.02,
    text="IM2 Launch",
    showarrow=False,
    # font=dict(color="blue", size=11)
)

fig.add_vline(
    x=IM2_LANDING_DATE,
    line_width=2,
    # line_dash="dashdot",
    # line_color="red"
)
fig.add_annotation(
    x=IM2_LANDING_DATE,
    yref="paper",
    y=1.02,
    text="IM2 Landing",
    showarrow=False,
    # font=dict(color="red", size=11)
)

fig.update_layout(
    xaxis_title='Remote Time (UTC)',
    yaxis_title='Temperature (°C)'
)
legend_mapping = {
    'temperature[0]': 'T0: enclosure by burnwire',
    'temperature[1]': 'T1: enclosure by mounting bracket'
}
for trace in fig.data:
    if trace.name in legend_mapping:
        trace.name = legend_mapping[trace.name]
fig.show()

## inter and extrapolating (experimental!)

In [ ]:
# 1. Ensure datetime‐type and UTC
IM2_THERMAL_SYSTEM_CHECK = "2025-03-06 23:32:51.009000+00:00"
IM2_LANDING_DATE           = pd.to_datetime(IM2_LANDING_DATE, utc=True)
IM2_THERMAL_SYSTEM_CHECK   = pd.to_datetime(IM2_THERMAL_SYSTEM_CHECK, utc=True)
YAOKI_LAST_TELEMETRY_DATE  = pd.to_datetime(YAOKI_LAST_TELEMETRY_DATE, utc=True)

# 2. Filter to post‐landing data
df_filtered = df_clean[df_clean['timestamp'] >= IM2_LANDING_DATE].copy()

fig = go.Figure()
poly_degree = 3

# Compute reference seconds
end_seconds   = (YAOKI_LAST_TELEMETRY_DATE - IM2_LANDING_DATE).total_seconds()
check_seconds = (IM2_THERMAL_SYSTEM_CHECK - IM2_LANDING_DATE).total_seconds()

# 3. Pick a consistent color for each sensor
colorway = pio.templates['simple_white'].layout.colorway
sensor_list = ['Temperature0', 'Temperature1']
sensor_color_map = {
    sensor_list[i]: colorway[i % len(colorway)]
    for i in range(len(sensor_list))
}

for sensor in sensor_list:
    df_sensor = df_filtered[df_filtered['sensor_name'] == sensor].copy()
    if df_sensor.empty:
        continue

    # Convert timestamps to seconds since landing
    df_sensor['seconds_since_landing'] = (
        df_sensor['timestamp'] - IM2_LANDING_DATE
    ).dt.total_seconds()

    x_all = df_sensor['seconds_since_landing'].values
    y_all = df_sensor['value'].values

    # — Polynomial fit (no extrapolation) —
    coeffs_poly = np.polyfit(x_all, y_all, deg=poly_degree)
    poly_fn     = np.poly1d(coeffs_poly)

    last_data_x = x_all.max()
    x_poly_fit  = np.linspace(x_all.min(), last_data_x, 100)
    y_poly_fit  = poly_fn(x_poly_fit)

    timestamps_poly = [
        IM2_LANDING_DATE + pd.Timedelta(seconds=float(sec)) 
        for sec in x_poly_fit
    ]

    # — Linear fit (post‐check extrapolate) —
    df_check = df_sensor[df_sensor['seconds_since_landing'] >= check_seconds]
    if not df_check.empty:
        x_check = df_check['seconds_since_landing'].values
        y_check = df_check['value'].values

        coeffs_lin = np.polyfit(x_check, y_check, deg=1)
        lin_fn     = np.poly1d(coeffs_lin)

        x_lin_fit = np.linspace(check_seconds, end_seconds, 100)
        y_lin_fit = lin_fn(x_lin_fit)

        timestamps_lin = [
            IM2_LANDING_DATE + pd.Timedelta(seconds=float(sec)) 
            for sec in x_lin_fit
        ]

    # Choose the sensor's color
    color = sensor_color_map[sensor]

    # — Plot raw scatter (same color for markers) —
    fig.add_trace(go.Scatter(
        x=df_sensor['timestamp'],
        y=df_sensor['value'],
        mode='markers',
        name=f'{sensor} Data',
        marker=dict(color=color, symbol='circle', size=6)
    ))

    # — Plot polynomial fit (same color) —
    fig.add_trace(go.Scatter(
        x=timestamps_poly,
        y=y_poly_fit,
        mode='lines',
        line=dict(color=color, dash='dash', width=2),
        name=f'{sensor} Poly Fit (deg={poly_degree})'
    ))

    # — Plot linear fit (same color) —
    if not df_check.empty:
        fig.add_trace(go.Scatter(
            x=timestamps_lin,
            y=y_lin_fit,
            mode='lines',
            line=dict(color=color, dash='solid', width=2),
            name=f'{sensor} Linear Fit (post‐check)'
        ))

# 4. Add shaded YAOKI OPS rectangle and label
fig.add_vrect(
    x0=YAOKI_FIRST_TELEMETRY_DATE,
    x1=YAOKI_LAST_TELEMETRY_DATE,
    fillcolor="LightGray",
    opacity=0.3,
    layer="below",
    line_width=0,
)

midpoint = (
    pd.to_datetime(YAOKI_FIRST_TELEMETRY_DATE).value +
    pd.to_datetime(YAOKI_LAST_TELEMETRY_DATE).value
) / 2
mid_datetime = pd.to_datetime(midpoint)

fig.add_annotation(
    x=mid_datetime,
    yref="paper",
    y=0.95,
    text="YAOKI surface OPS",
    showarrow=False,
    font=dict(color="black", size=12),
    bgcolor="rgba(255,255,255,0.7)"
)
fig.add_vline(
    x=IM2_LANDING_DATE,
    line_width=2,
)
fig.add_annotation(
    x=IM2_LANDING_DATE,
    yref="paper",
    y=1.02,
    text="IM2 Landing",
    showarrow=False,
)
# 5. Final layout tweaks
fig.update_layout(
    title='Temperature Fits: Polynomial (to last point) & Linear (post‐check extrapolate)',
    xaxis_title='Timestamp (UTC)',
    yaxis_title='Temperature (°C)',
    template='simple_white',
    legend_title='Series'
)
fig.show()
